In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import cv2

In [2]:
import sys
import os

# 1. Get the current directory (Minor Project/Notebook)
# 2. Get the parent directory (Minor Project)
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))

# 3. Add that parent directory to sys.path
if root_path not in sys.path:
    sys.path.append(root_path)

In [3]:
from Models.dataloader import CustomDataset,DataLoader

In [4]:
class DepthEncoder(nn.Module):
    def __init__(self,in_channel = 2, embeddings_channel = 256):
        super().__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channel,64,kernel_size=3,padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2,inplace=True)
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2,inplace=True)
        )

        self.layer3 = nn.Sequential(
            nn.Conv2d(128,256,kernel_size=3,padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2,inplace=True)
        )

        self.final_projection = nn.Conv2d(256,embeddings_channel,kernel_size=1)


    def forward(self,x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        return self.final_projection(x)

In [5]:
class PositionalEmbeddings(nn.Module):
    def __init__(self,num_tokens = 4096,embedding_dim = 256):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1,num_tokens,embedding_dim) * 0.02)

    def forward(self,x):
        return x + self.pos_embedding
    
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=256, nhead=8):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

    def forward(self, x):
        return self.transformer(x)

In [6]:
class DepthSAMFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.pos_encoder = PositionalEmbeddings()
        self.fusion_transformer = TransformerBlock()

        self.output_conv = nn.Conv2d(256,256,kernel_size=1)

    def forward(self,img_feature,depth_feature):
        # Features has shape [B,256,64,64]
        B,C,H,W = img_feature.shape

        # Flatten should start from dimension 2 and we get result in shape [B,256,4096]
        # then we reshape it to [B,4096,256]
        img_tokens = img_feature.flatten(2).permute(0,2,1)
        depth_tokens = depth_feature.flatten(2).permute(0,2,1)

        img_tokens = self.pos_encoder(img_tokens)
        depth_tokens = self.pos_encoder(depth_tokens)

        fused_tokens = self.fusion_transformer(img_tokens+depth_tokens)

        fused_grid = fused_tokens.permute(0,2,1).reshape(B,C,H,W)

        return self.output_conv(fused_grid)


